# Bias in Bios Concept-QA

Train the reusable Concept-QA module for `LabHC/bias_in_bios`.

Main choices:

- input encoder: `sentence-transformers/all-MiniLM-L6-v2`
- query vocabulary: fixed BIOS concept set
- supervision: calibrated description-similarity soft targets
- artifact: small Concept-QA checkpoint and optional split-level cache

In [ ]:
# %pip install -e ..
# %pip install datasets sentence-transformers scikit-learn -q

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from staq.config import default_paths, detect_device
from staq.models import ConceptNet2
from staq.training import seed_everything

In [2]:
def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    concept_relpath = Path("assets") / "concepts" / "bias_in_bios.csv"

    # First handle the normal cases: kernel is in the repo root or notebooks/.
    for candidate in [start_path, *start_path.parents]:
        if (candidate / concept_relpath).exists():
            return candidate

    # Workbench kernels may start in /home/jupyter while the repo is a child directory.
    likely_roots = [start_path, Path.home(), Path("/home/jupyter"), Path("/workspace"), Path("/workspaces")]
    seen = set()
    for base in likely_roots:
        try:
            base = base.resolve()
        except OSError:
            continue
        if base in seen or not base.exists():
            continue
        seen.add(base)
        try:
            matches = list(base.glob(f"*/{concept_relpath.as_posix()}"))
            matches += list(base.glob(f"*/*/{concept_relpath.as_posix()}"))
        except OSError:
            continue
        if matches:
            return matches[0].parents[2]

    raise FileNotFoundError(
        "Could not find assets/concepts/bias_in_bios.csv. "
        f"Kernel working directory: {start_path}. "
        "Set repo_root manually to the server checkout path if the repo is mounted elsewhere."
    )


manual_repo_root = None  # Set to the server checkout path if auto-detection fails.
repo_root = Path(manual_repo_root).expanduser().resolve() if manual_repo_root else find_repo_root(Path.cwd())
paths = default_paths(repo_root)
paths.ensure_artifact_dirs()

device = detect_device()
random_seed = 13
seed_everything(random_seed)

encoder_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_dim = 384

dataset_name = "LabHC/bias_in_bios"
text_column = "hard_text"
target_column = "profession"
sensitive_column = "gender"

profession_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

gender_names = ["male", "female"]

{
    "kernel_cwd": str(Path.cwd()),
    "repo_root": str(repo_root),
    "device": str(device),
    "encoder_name": encoder_name,
    "embedding_dim": embedding_dim,
    "dataset_name": dataset_name,
}

{'kernel_cwd': '/home/jupyter/staq/notebooks',
 'repo_root': '/home/jupyter/staq',
 'device': 'cuda',
 'encoder_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_dim': 384,
 'dataset_name': 'LabHC/bias_in_bios'}

In [3]:
raw_train = load_dataset(dataset_name, split="train")
raw_dev = load_dataset(dataset_name, split="dev")
raw_test = load_dataset(dataset_name, split="test")

{
    "train": len(raw_train),
    "dev": len(raw_dev),
    "test": len(raw_test),
    "columns": raw_train.column_names,
}

{'train': 257478,
 'dev': 39642,
 'test': 99069,
 'columns': ['hard_text', 'profession', 'gender']}

In [4]:
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
if not concepts_path.exists():
    raise FileNotFoundError(f"Concept file not found: {concepts_path}")

concepts_df = pd.read_csv(concepts_path)
required_concept_columns = {"concept", "kind", "description"}
missing_concept_columns = required_concept_columns.difference(concepts_df.columns)
if missing_concept_columns:
    raise ValueError(
        f"Concept file is missing columns {sorted(missing_concept_columns)}. "
        f"Loaded columns: {concepts_df.columns.tolist()} from {concepts_path}"
    )

concept_names = concepts_df["concept"].tolist()
concept_texts = concepts_df["description"].tolist()
sensitive_concepts = concepts_df.query("kind != 'utility'")["concept"].tolist()
sensitive_mask = torch.tensor(concepts_df["kind"].ne("utility").to_numpy(), dtype=torch.bool)

{
    "repo_root": str(repo_root),
    "concepts_path": str(concepts_path),
    "concept_columns": concepts_df.columns.tolist(),
    "num_concepts": len(concepts_df),
    "by_kind": concepts_df["kind"].value_counts().to_dict(),
    "num_sensitive_concepts": int(sensitive_mask.sum()),
}

{'repo_root': '/home/jupyter/staq',
 'concepts_path': '/home/jupyter/staq/assets/concepts/bias_in_bios.csv',
 'concept_columns': ['concept', 'kind', 'description'],
 'num_concepts': 39,
 'by_kind': {'utility': 33, 'proxy_sensitive': 4, 'direct_sensitive': 2},
 'num_sensitive_concepts': 6}

In [5]:
encoder = SentenceTransformer(encoder_name, device=str(device))

concept_embeddings = encoder.encode(
    concept_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(device)

concept_dictionary = concept_embeddings.T.contiguous()

{
    "concept_embeddings": tuple(concept_embeddings.shape),
    "concept_dictionary": tuple(concept_dictionary.shape),
}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'concept_embeddings': (39, 384), 'concept_dictionary': (384, 39)}

In [6]:
sample_rows = raw_train.select(range(8))
sample_texts = [row[text_column] for row in sample_rows]

sample_embeddings = encoder.encode(
    sample_texts,
    batch_size=8,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).to(device)

sample_concept_qa_inputs = torch.cat(
    [
        sample_embeddings.unsqueeze(1).repeat(1, len(concept_names), 1),
        concept_embeddings.unsqueeze(0).repeat(len(sample_texts), 1, 1),
    ],
    dim=2,
).reshape(len(sample_texts) * len(concept_names), -1)

{
    "sample_embeddings": tuple(sample_embeddings.shape),
    "concept_qa_input_shape": tuple(sample_concept_qa_inputs.shape),
    "expected_concept_qa_input_dim": embedding_dim * 2,
}

{'sample_embeddings': (8, 384),
 'concept_qa_input_shape': (312, 768),
 'expected_concept_qa_input_dim': 768}

In [7]:
pd.DataFrame(
    {
        "profession": [profession_names[row[target_column]] for row in sample_rows],
        "gender": [gender_names[row[sensitive_column]] for row in sample_rows],
        "text": sample_texts,
    }
)

,profession,gender,text
0,professor,male,He is also the project lead of and major contr...
1,nurse,female,"She is able to assess, diagnose and treat mino..."
2,attorney,female,"Prior to law school, Brittni graduated magna c..."
3,journalist,male,He regularly contributes to India’s First Onli...
4,professor,male,He completed his medical degree at Northwester...
5,attorney,male,Steve has practiced law in Kentucky for over 3...
6,professor,male,"His research is in information archiving, anal..."
7,poet,female,"Trained as a doctor, she lives in Algiers wher..."


## Qwen Teacher Probe

Small yes/no LLM check before generating full Concept-QA targets.

In [8]:
qwen_teacher_name = "Qwen/Qwen3-1.7B"
qwen_probe_size = 128
qwen_batch_size = 4
qwen_max_length = 1024
qwen_max_new_tokens = 8
qwen_full_dataset_size = 257_478

import gc
import time

if globals().get("qwen_loaded_teacher_name") != qwen_teacher_name:
    globals().pop("qwen_model", None)
    globals().pop("qwen_tokenizer", None)
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_teacher_name)
    qwen_tokenizer.padding_side = "left"
    qwen_tokenizer.pad_token = qwen_tokenizer.pad_token or qwen_tokenizer.eos_token

    model_kwargs = {"torch_dtype": torch.float16} if device.type == "cuda" else {}
    qwen_model = AutoModelForCausalLM.from_pretrained(qwen_teacher_name, **model_kwargs).to(device)
    qwen_model.eval()
    qwen_model.generation_config.pad_token_id = qwen_tokenizer.pad_token_id
    qwen_loaded_teacher_name = qwen_teacher_name


def build_qwen_teacher_prompt(description, text):
    description = description.strip().rstrip(".")
    question = description.replace("this biography mentions", "Does the biography mention", 1)
    question = question.replace("this biography uses", "Does the biography use", 1)
    question = question.replace("this biography emphasizes", "Does the biography emphasize", 1)
    question = f"{question}?"
    messages = [
        {"role": "system", "content": "Answer yes/no questions about biographies using only the biography text."},
        {
            "role": "user",
            "content": (
                f"{question}\n\n"
                f"Biography: {text}\n\n"
                "Answer only Yes or No."
            ),
        },
    ]
    chat_template_kwargs = {
        "tokenize": False,
        "add_generation_prompt": True,
    }
    if "Qwen3" in qwen_teacher_name:
        chat_template_kwargs["enable_thinking"] = False
    return qwen_tokenizer.apply_chat_template(messages, **chat_template_kwargs)


def parse_qwen_answer(answer):
    cleaned = answer.strip().strip(" \"'`")
    first_token = cleaned.split(maxsplit=1)[0].strip(":.,").lower() if cleaned else ""
    if first_token == "yes":
        return 1.0, "Yes"
    if first_token == "no":
        return 0.0, "No"
    return float("nan"), "Unparsed"


qwen_probe_rows = raw_train.select(range(qwen_probe_size))
qwen_probe_texts = [row[text_column] for row in qwen_probe_rows]
qwen_probe_embeddings = encoder.encode(
    qwen_probe_texts,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(device)

qwen_prompt_rows = []
for row_index, text in enumerate(qwen_probe_texts):
    for concept_index, description in enumerate(concept_texts):
        qwen_prompt_rows.append(
            {
                "row": row_index,
                "concept_index": concept_index,
                "prompt": build_qwen_teacher_prompt(description, text),
            }
        )

qwen_outputs = []
qwen_start_time = time.perf_counter()
with torch.inference_mode():
    for start in range(0, len(qwen_prompt_rows), qwen_batch_size):
        batch_rows = qwen_prompt_rows[start : start + qwen_batch_size]
        encoded = qwen_tokenizer(
            [row["prompt"] for row in batch_rows],
            padding=True,
            truncation=True,
            max_length=qwen_max_length,
            return_tensors="pt",
        ).to(device)
        generated = qwen_model.generate(
            **encoded,
            max_new_tokens=qwen_max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.pad_token_id,
            eos_token_id=qwen_tokenizer.eos_token_id,
        )
        generated_answers = qwen_tokenizer.batch_decode(
            generated[:, encoded["input_ids"].shape[1] :],
            skip_special_tokens=True,
        )

        for row, answer in zip(batch_rows, generated_answers):
            qwen_yes_score, answer_label = parse_qwen_answer(answer)
            qwen_outputs.append(
                {
                    "row": row["row"],
                    "concept_index": row["concept_index"],
                    "answer": answer.strip(),
                    "answer_label": answer_label,
                    "qwen_yes_score": qwen_yes_score,
                }
            )

        del encoded, generated
        if device.type == "cuda":
            torch.cuda.empty_cache()

qwen_elapsed_seconds = time.perf_counter() - qwen_start_time
qwen_num_prompts = len(qwen_prompt_rows)
qwen_prompts_per_second = qwen_num_prompts / qwen_elapsed_seconds
qwen_estimated_full_prompts = qwen_full_dataset_size * len(concept_names)
qwen_estimated_full_hours = qwen_estimated_full_prompts / qwen_prompts_per_second / 3600
print(
    f"Qwen benchmark: {len(qwen_probe_texts)} biographies x {len(concept_names)} concepts "
    f"= {qwen_num_prompts:,} prompts in {qwen_elapsed_seconds / 60:.1f} minutes "
    f"({qwen_prompts_per_second:.2f} prompts/sec)."
)
print(
    f"Estimated full run: {qwen_estimated_full_prompts:,} prompts "
    f"≈ {qwen_estimated_full_hours:.1f} hours ({qwen_estimated_full_hours / 24:.1f} days)."
)

qwen_probe = pd.DataFrame(qwen_outputs)
qwen_yes_scores = torch.full((len(qwen_probe_texts), len(concept_names)), float("nan"), dtype=torch.float32)
for output in qwen_outputs:
    qwen_yes_scores[output["row"], output["concept_index"]] = output["qwen_yes_score"]

bi_encoder_scores = (qwen_probe_embeddings @ concept_embeddings.T).detach().float().cpu()


def show_qwen_teacher_concepts(row_index=0):
    row_outputs = qwen_probe[qwen_probe["row"].eq(row_index)].copy()
    row_outputs["concept"] = [concept_names[idx] for idx in row_outputs["concept_index"]]
    row_outputs["kind"] = [concepts_df.loc[idx, "kind"] for idx in row_outputs["concept_index"]]
    row_outputs["description"] = [concept_texts[idx] for idx in row_outputs["concept_index"]]
    row_outputs["bi_encoder_similarity"] = [
        float(bi_encoder_scores[row_index, idx]) for idx in row_outputs["concept_index"]
    ]
    print(f"profession: {profession_names[qwen_probe_rows[row_index][target_column]]}")
    print(f"gender: {gender_names[qwen_probe_rows[row_index][sensitive_column]]}")
    print(qwen_probe_texts[row_index])
    return row_outputs.sort_values(
        ["qwen_yes_score", "concept"], ascending=[False, True]
    )[
        [
            "concept",
            "kind",
            "answer_label",
            "qwen_yes_score",
            "bi_encoder_similarity",
            "description",
            "answer",
        ]
    ]


show_qwen_teacher_concepts(0)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen benchmark: 128 biographies x 39 concepts = 4,992 prompts in 3.9 minutes (21.51 prompts/sec).
Estimated full run: 10,041,642 prompts ≈ 129.6 hours (5.4 days).
profession: professor
gender: male
He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates


,concept,kind,answer_label,qwen_yes_score,bi_encoder_similarity,description,answer
13,advanced_degree_or_training,utility,Yes,1.0,0.246230,"this biography mentions graduate degrees, doct...",Yes
31,business_or_client_service,utility,Yes,1.0,0.120433,"this biography mentions clients, consulting, b...",Yes
10,classroom_instruction,utility,Yes,1.0,0.166538,"this biography mentions classroom teaching, pu...",Yes.
14,software_development,utility,Yes,1.0,0.504486,"this biography mentions writing software, prog...",Yes
11,university_affiliation,utility,Yes,1.0,0.211494,"this biography mentions universities, departme...",Yes
18,visual_art_practice,utility,Yes,1.0,0.142675,"this biography mentions painting, drawing, scu...",Yes.
37,appearance_or_body_presentation,proxy_sensitive,No,0.0,0.084751,"this biography emphasizes appearance, beauty, ...",No.
16,building_or_spatial_design,utility,No,0.0,0.262476,"this biography mentions buildings, architectur...",No.
19,camera_or_photography_work,utility,No,0.0,0.167027,"this biography mentions photography, cameras, ...",No.
2,clinical_care,utility,No,0.0,0.102474,"this biography mentions direct clinical care, ...",No.


In [9]:
# Qwen probe diagnostics: keep this lightweight while we iterate on one or a few biographies.
qwen_probe_review = qwen_probe.copy()
qwen_probe_review["concept"] = [concept_names[idx] for idx in qwen_probe_review["concept_index"]]
qwen_probe_review["kind"] = [concepts_df.loc[idx, "kind"] for idx in qwen_probe_review["concept_index"]]
qwen_probe_review["description"] = [concept_texts[idx] for idx in qwen_probe_review["concept_index"]]
qwen_probe_review["profession"] = [
    profession_names[qwen_probe_rows[row_index][target_column]]
    for row_index in qwen_probe_review["row"]
]
qwen_probe_review["gender"] = [
    gender_names[qwen_probe_rows[row_index][sensitive_column]]
    for row_index in qwen_probe_review["row"]
]
qwen_probe_review["text"] = [qwen_probe_texts[row_index] for row_index in qwen_probe_review["row"]]
qwen_probe_review["bi_encoder_similarity"] = [
    float(bi_encoder_scores[row_index, concept_index])
    for row_index, concept_index in zip(qwen_probe_review["row"], qwen_probe_review["concept_index"])
]

qwen_concept_summary = (
    qwen_probe_review.groupby(["concept", "kind"], as_index=False)
    .agg(
        yes_count=("qwen_yes_score", "sum"),
        yes_rate=("qwen_yes_score", "mean"),
        mean_bi_encoder_similarity=("bi_encoder_similarity", "mean"),
        description=("description", "first"),
    )
    .sort_values(["yes_rate", "yes_count", "concept"], ascending=[False, False, True])
)
qwen_concept_summary["yes_count"] = qwen_concept_summary["yes_count"].astype(int)

qwen_positive_concepts_by_row = (
    qwen_probe_review[qwen_probe_review["qwen_yes_score"].eq(1.0)]
    .groupby("row")["concept"]
    .apply(lambda concepts: ", ".join(concepts))
)
qwen_row_summary = (
    qwen_probe_review.groupby(["row", "profession", "gender"], as_index=False)
    .agg(
        yes_count=("qwen_yes_score", "sum"),
        yes_rate=("qwen_yes_score", "mean"),
        text=("text", "first"),
    )
    .sort_values(["yes_count", "row"], ascending=[False, True])
)
qwen_row_summary["yes_count"] = qwen_row_summary["yes_count"].astype(int)
qwen_row_summary["positive_concepts"] = qwen_row_summary["row"].map(qwen_positive_concepts_by_row).fillna("")

positive_sensitive_proxy = qwen_probe_review[
    qwen_probe_review["kind"].ne("utility") & qwen_probe_review["qwen_yes_score"].eq(1.0)
].sort_values(["row", "concept"])

qwen_positive_low_encoder = qwen_probe_review[
    qwen_probe_review["qwen_yes_score"].eq(1.0)
    & qwen_probe_review["bi_encoder_similarity"].lt(0.20)
].sort_values(["bi_encoder_similarity", "row", "concept"])

print(f"Qwen probe rows: {len(qwen_probe_texts)} biographies x {len(concept_names)} concepts")
display(qwen_concept_summary)
display(qwen_row_summary[["row", "profession", "gender", "yes_count", "yes_rate", "positive_concepts", "text"]])
display(positive_sensitive_proxy[["row", "profession", "gender", "concept", "kind", "answer_label", "answer", "text"]])
display(qwen_positive_low_encoder[["row", "profession", "gender", "concept", "kind", "answer_label", "answer", "bi_encoder_similarity", "description", "text"]].head(30))

Qwen probe rows: 128 biographies x 39 concepts


,concept,kind,yes_count,yes_rate,mean_bi_encoder_similarity,description
3,business_or_client_service,utility,61,0.476562,0.184688,"this biography mentions clients, consulting, b..."
37,university_affiliation,utility,58,0.453125,0.192056,"this biography mentions universities, departme..."
38,visual_art_practice,utility,46,0.359375,0.143887,"this biography mentions painting, drawing, scu..."
0,advanced_degree_or_training,utility,40,0.312500,0.238956,"this biography mentions graduate degrees, doct..."
5,classroom_instruction,utility,31,0.242188,0.158135,"this biography mentions classroom teaching, pu..."
32,research_publications,utility,15,0.117188,0.204023,"this biography mentions research publications,..."
20,literary_writing,utility,14,0.109375,0.154818,"this biography mentions poetry, books, literar..."
26,nursing_care,utility,12,0.093750,0.221473,"this biography mentions nursing care, nursing ..."
29,public_speaking_or_events,utility,11,0.085938,0.171789,"this biography mentions talks, conferences, le..."
6,clinical_care,utility,10,0.078125,0.215751,"this biography mentions direct clinical care, ..."


,row,profession,gender,yes_count,yes_rate,positive_concepts,text
103,103,professor,female,10,0.256410,"classroom_instruction, university_affiliation,...",She specializes in pedagogy (ways of teaching ...
55,55,yoga_teacher,female,9,0.230769,"classroom_instruction, visual_art_practice, li...",She combines absolute firmness in her teaching...
16,16,professor,female,8,0.205128,"classroom_instruction, university_affiliation,...","In this capacity, Cara teaches courses in urba..."
113,113,professor,female,8,0.205128,"medical_training, classroom_instruction, unive...",She has been trained by professional artist an...
49,49,professor,male,7,0.179487,"classroom_instruction, university_affiliation,...",His research considers the intersection of hum...
...,...,...,...,...,...,...,...
78,78,professor,female,0,0.000000,,She studies marine endangered species and inte...
82,82,professor,male,0,0.000000,,"His main research field is molecular modeling,..."
84,84,photographer,male,0,0.000000,,Through his independent work and collaboration...
85,85,professor,female,0,0.000000,,She specializes in the analysis of processes o...


,row,profession,gender,concept,kind,answer_label,answer,text
620,15,journalist,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She graduated in May 2008 from Corban College ...
2453,62,attorney,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She is a Northern California native and is ded...
2531,64,nurse,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She received her Associates degree from Northe...
2609,66,journalist,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She covered Hurricane Matthew and several nota...
3505,89,physician,female,gendered_titles,direct_sensitive,Yes,Yes.,"Ms. Dziedzic practices medicine in Madison, CT..."
3817,97,physician,female,gendered_titles,direct_sensitive,Yes,Yes.,"Ms. Deslauriers practices medicine in Corona, ..."
4052,103,professor,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She specializes in pedagogy (ways of teaching ...
4053,103,professor,female,parenting_or_caregiving_mentions,proxy_sensitive,Yes,Yes.,She specializes in pedagogy (ways of teaching ...


,row,profession,gender,concept,kind,answer_label,answer,bi_encoder_similarity,description,text
3918,100,filmmaker,male,visual_art_practice,utility,Yes,Yes.,-0.002339,"this biography mentions painting, drawing, scu...",His first work in the horror genre was writing...
408,10,teacher,female,visual_art_practice,utility,Yes,Yes.,0.007958,"this biography mentions painting, drawing, scu...",When comparing Jim Giordano's ratings to other...
3255,83,accountant,female,visual_art_practice,utility,Yes,Yes.,0.020624,"this biography mentions painting, drawing, scu...",David W. Overby has been issued a Mississippi ...
4464,114,physician,male,visual_art_practice,utility,Yes,Yes.,0.022476,"this biography mentions painting, drawing, scu...",He has a 4.5 out of 5 star average patient rat...
3840,98,filmmaker,male,visual_art_practice,utility,Yes,Yes.,0.026324,"this biography mentions painting, drawing, scu...","His feature, Outtake Reel (2011), is a unique ..."
3365,86,nurse,female,university_affiliation,utility,Yes,Yes,0.034284,"this biography mentions universities, departme...",She graduated with honors in 1979. Having more...
3841,98,filmmaker,male,camera_or_photography_work,utility,Yes,Yes.,0.040275,"this biography mentions photography, cameras, ...","His feature, Outtake Reel (2011), is a unique ..."
1117,28,composer,female,comedy_or_stage_performance,utility,Yes,Yes.,0.045144,"this biography mentions comedy, stand-up, stag...","These twisted lines and curlicue exploits, hea..."
4144,106,nurse,female,classroom_instruction,utility,Yes,Yes.,0.050064,"this biography mentions classroom teaching, pu...",Marci has worked in the Breast Field for 20 pl...
3658,93,professor,male,business_or_client_service,utility,Yes,Yes.,0.056904,"this biography mentions clients, consulting, b...","This article is drawn from his new book, Virgi..."


In [10]:
print(concepts_df.loc[concepts_df["concept"].eq("direct_gender_pronouns"), "description"].iloc[0])
print(concepts_df[["concept", "description"]].tail(8))

this biography uses gendered personal pronouns such as he, him, his, she, her, or hers
                             concept  \
31        business_or_client_service   
32      financial_or_accounting_work   
33            direct_gender_pronouns   
34                   gendered_titles   
35             gendered_family_roles   
36  parenting_or_caregiving_mentions   
37   appearance_or_body_presentation   
38    gendered_organization_or_award   

                                          description  
31  this biography mentions clients, consulting, b...  
32  this biography mentions accounting, taxes, pay...  
33  this biography uses gendered personal pronouns...  
34  this biography uses explicitly gendered titles...  
35  this biography mentions a spouse, parent, or c...  
36  this biography mentions parenting, children, c...  
37  this biography emphasizes appearance, beauty, ...  
38  this biography mentions gendered organizations...  


In [11]:
sensitive_indices = concepts_df.index[concepts_df["kind"].ne("utility")].to_numpy()

sensitive_probe_rows = []
for row_index in range(len(qwen_probe_texts)):
    row_answers = qwen_probe[qwen_probe["row"].eq(row_index)].set_index("concept_index")
    for concept_index in sensitive_indices:
        sensitive_probe_rows.append(
            {
                "row": row_index,
                "profession": profession_names[qwen_probe_rows[row_index][target_column]],
                "gender": gender_names[qwen_probe_rows[row_index][sensitive_column]],
                "concept": concept_names[concept_index],
                "kind": concepts_df.loc[concept_index, "kind"],
                "answer_label": row_answers.loc[concept_index, "answer_label"],
                "answer": row_answers.loc[concept_index, "answer"],
                "qwen_yes_score": float(qwen_yes_scores[row_index, concept_index]),
                "bi_encoder_similarity": float(bi_encoder_scores[row_index, concept_index]),
                "text": qwen_probe_texts[row_index],
            }
        )

sensitive_probe = pd.DataFrame(sensitive_probe_rows)
sensitive_score_matrix = sensitive_probe.pivot_table(
    index=["row", "profession", "gender"],
    columns="concept",
    values="qwen_yes_score",
).reset_index()

positive_sensitive_scores = sensitive_probe[
    sensitive_probe["qwen_yes_score"].eq(1.0)
].sort_values(["row", "concept"])

display(sensitive_score_matrix)
display(positive_sensitive_scores[["row", "profession", "gender", "concept", "kind", "answer_label", "answer", "text"]])

concept,row,profession,gender,appearance_or_body_presentation,direct_gender_pronouns,gendered_family_roles,gendered_organization_or_award,gendered_titles,parenting_or_caregiving_mentions
0,0,professor,male,0.0,0.0,0.0,0.0,0.0,0.0
1,1,nurse,female,0.0,0.0,0.0,0.0,0.0,0.0
2,2,attorney,female,0.0,0.0,0.0,0.0,0.0,0.0
3,3,journalist,male,0.0,0.0,0.0,0.0,0.0,0.0
4,4,professor,male,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
123,123,photographer,female,0.0,0.0,0.0,0.0,0.0,0.0
124,124,professor,male,0.0,0.0,0.0,0.0,0.0,0.0
125,125,yoga_teacher,female,0.0,0.0,0.0,0.0,0.0,0.0
126,126,teacher,female,0.0,0.0,0.0,0.0,0.0,0.0


,row,profession,gender,concept,kind,answer_label,answer,text
92,15,journalist,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She graduated in May 2008 from Corban College ...
374,62,attorney,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She is a Northern California native and is ded...
386,64,nurse,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She received her Associates degree from Northe...
398,66,journalist,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She covered Hurricane Matthew and several nota...
535,89,physician,female,gendered_titles,direct_sensitive,Yes,Yes.,"Ms. Dziedzic practices medicine in Madison, CT..."
583,97,physician,female,gendered_titles,direct_sensitive,Yes,Yes.,"Ms. Deslauriers practices medicine in Corona, ..."
620,103,professor,female,gendered_family_roles,proxy_sensitive,Yes,Yes.,She specializes in pedagogy (ways of teaching ...
621,103,professor,female,parenting_or_caregiving_mentions,proxy_sensitive,Yes,Yes.,She specializes in pedagogy (ways of teaching ...


In [ ]:
encoding_batch_size = 256
raw_splits = {
    "train": raw_train,
    "dev": raw_dev,
    "test": raw_test,
}


def encode_bios_split(split):
    texts = [row[text_column] for row in split]
    embeddings = encoder.encode(
        texts,
        batch_size=encoding_batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).to(device)
    professions = [profession_names[row[target_column]] for row in split]
    genders = [gender_names[row[sensitive_column]] for row in split]
    return {
        "texts": texts,
        "embeddings": embeddings,
        "similarity_scores": embeddings @ concept_embeddings.T,
        "profession_labels": professions,
        "gender_labels": genders,
        "profession_ids": torch.tensor([row[target_column] for row in split], dtype=torch.long),
        "gender_ids": torch.tensor([row[sensitive_column] for row in split], dtype=torch.long),
    }


encoded_splits = {
    split_name: encode_bios_split(split)
    for split_name, split in raw_splits.items()
}

{
    split_name: {
        "embeddings": tuple(split_data["embeddings"].shape),
        "similarity_scores": tuple(split_data["similarity_scores"].shape),
    }
    for split_name, split_data in encoded_splits.items()
}

In [ ]:
train_similarity_scores = encoded_splits["train"]["similarity_scores"]
concept_medians = train_similarity_scores.median(dim=0).values
concept_stds = train_similarity_scores.std(dim=0).clamp_min(1e-6)
calibration_temperature = 1.5


def calibrate_similarity_scores(similarity_scores):
    return torch.sigmoid(
        (similarity_scores - concept_medians.unsqueeze(0))
        / (calibration_temperature * concept_stds.unsqueeze(0))
    )


for split_data in encoded_splits.values():
    split_data["calibrated_soft_targets"] = calibrate_similarity_scores(split_data["similarity_scores"])

train_targets = encoded_splits["train"]["calibrated_soft_targets"]
calibrated_summary = pd.DataFrame(
    {
        "concept": concept_names,
        "kind": concepts_df["kind"],
        "mean": train_targets.mean(dim=0).cpu().numpy(),
        "std": train_targets.std(dim=0).cpu().numpy(),
        "p05": torch.quantile(train_targets, 0.05, dim=0).cpu().numpy(),
        "p50": torch.quantile(train_targets, 0.50, dim=0).cpu().numpy(),
        "p95": torch.quantile(train_targets, 0.95, dim=0).cpu().numpy(),
    }
)

calibrated_summary.sort_values("mean", ascending=False)

In [ ]:
def show_top_calibrated_concepts(row_index, split_name="train", top_k=8):
    split_data = encoded_splits[split_name]
    scores = split_data["calibrated_soft_targets"][row_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    print(f"profession: {split_data['profession_labels'][row_index]}")
    print(f"gender: {split_data['gender_labels'][row_index]}")
    print(split_data["texts"][row_index])
    return pd.DataFrame(
        {
            "concept": [concept_names[idx] for idx in top_indices],
            "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
            "score": scores[top_indices],
        }
    )


show_top_calibrated_concepts(0)

In [ ]:
def show_high_calibrated_examples(concept, split_name="train", top_k=5):
    split_data = encoded_splits[split_name]
    concept_index = concept_names.index(concept)
    scores = split_data["calibrated_soft_targets"][:, concept_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame(
        {
            "score": scores[top_indices],
            "profession": [split_data["profession_labels"][idx] for idx in top_indices],
            "gender": [split_data["gender_labels"][idx] for idx in top_indices],
            "text": [split_data["texts"][idx] for idx in top_indices],
        }
    )


show_high_calibrated_examples("software_development")

In [ ]:
batch_size = 256

def make_concept_qa_dataset(split_name):
    split_data = encoded_splits[split_name]
    return TensorDataset(
        split_data["embeddings"].cpu(),
        split_data["calibrated_soft_targets"].cpu(),
    )


train_dataset = make_concept_qa_dataset("train")
dev_dataset = make_concept_qa_dataset("dev")
test_dataset = make_concept_qa_dataset("test")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

{
    "train_examples": len(train_dataset),
    "dev_examples": len(dev_dataset),
    "test_examples": len(test_dataset),
    "num_concepts": len(concept_names),
}

In [ ]:
def build_text_concept_qa_inputs(text_embeddings, concept_embeddings):
    batch_size = text_embeddings.size(0)
    num_concepts = concept_embeddings.size(0)
    text_rep = text_embeddings.unsqueeze(1).repeat(1, num_concepts, 1)
    concept_rep = concept_embeddings.unsqueeze(0).repeat(batch_size, 1, 1)
    return torch.cat([text_rep, concept_rep], dim=2).reshape(batch_size * num_concepts, -1)


def tensor_corrcoef(left, right):
    left = left.flatten()
    right = right.flatten()
    left = left - left.mean()
    right = right - right.mean()
    denom = left.norm() * right.norm()
    if float(denom) == 0.0:
        return torch.tensor(float("nan"), device=left.device)
    return (left @ right) / denom


def rank_tensor(values):
    order = values.flatten().argsort()
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(len(order), device=values.device, dtype=torch.float32)
    return ranks.view_as(values)


@torch.no_grad()
def evaluate_concept_qa_soft(model, loader):
    model.eval()
    losses = []
    hard_accuracies = []
    pred_batches = []
    target_batches = []

    for text_batch, target_batch in loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)
        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = model(inputs).view_as(target_batch)
        probs = torch.sigmoid(logits)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)
        preds = (probs >= 0.5).float()
        hard_targets = (target_batch >= 0.5).float()

        losses.append(float(loss.item()))
        hard_accuracies.append(float((preds == hard_targets).float().mean().item()))
        pred_batches.append(probs.detach().cpu())
        target_batches.append(target_batch.detach().cpu())

    all_preds = torch.cat(pred_batches)
    all_targets = torch.cat(target_batches)
    pearson = tensor_corrcoef(all_preds, all_targets)
    spearman = tensor_corrcoef(rank_tensor(all_preds), rank_tensor(all_targets))

    return {
        "loss": float(np.mean(losses)),
        "mae": float((all_preds - all_targets).abs().mean().item()),
        "rmse": float(torch.sqrt(((all_preds - all_targets) ** 2).mean()).item()),
        "pearson": float(pearson.item()),
        "spearman": float(spearman.item()),
        "hard_accuracy": float(np.mean(hard_accuracies)),
    }

In [ ]:
concept_qa_model = ConceptNet2(embed_dims=embedding_dim).to(device)
optimizer = torch.optim.AdamW(concept_qa_model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 5

concept_qa_history = []
for epoch in range(1, num_epochs + 1):
    concept_qa_model.train()
    train_losses = []

    for text_batch, target_batch in train_loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)

        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = concept_qa_model(inputs).view_as(target_batch)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    dev_metrics = evaluate_concept_qa_soft(concept_qa_model, dev_loader)
    concept_qa_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "dev_loss": dev_metrics["loss"],
            "dev_mae": dev_metrics["mae"],
            "dev_rmse": dev_metrics["rmse"],
            "dev_pearson": dev_metrics["pearson"],
            "dev_spearman": dev_metrics["spearman"],
            "dev_hard_accuracy": dev_metrics["hard_accuracy"],
        }
    )

concept_qa_test_metrics = evaluate_concept_qa_soft(concept_qa_model, test_loader)

pd.DataFrame(concept_qa_history), concept_qa_test_metrics

In [ ]:
@torch.no_grad()
def predict_concept_scores_for_row(row_index, split_name="dev"):
    concept_qa_model.eval()
    split_data = encoded_splits[split_name]
    text_embedding = split_data["embeddings"][row_index : row_index + 1].to(device)
    inputs = build_text_concept_qa_inputs(text_embedding, concept_embeddings)
    logits = concept_qa_model(inputs).view(1, len(concept_names))
    return torch.sigmoid(logits).squeeze(0).cpu().numpy()


row_index = 0
split_name = "dev"
model_scores = predict_concept_scores_for_row(row_index, split_name=split_name)
target_scores = encoded_splits[split_name]["calibrated_soft_targets"][row_index].cpu().numpy()
top_indices = np.argsort(model_scores)[::-1][:8]

pd.DataFrame(
    {
        "concept": [concept_names[idx] for idx in top_indices],
        "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
        "model_score": model_scores[top_indices],
        "target_score": target_scores[top_indices],
    }
)

In [ ]:
qa_experiment_name = "bias_in_bios_concept_qa_prompt_full"
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_history.json"
qa_cache_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_cache.pt"
save_split_cache = False

target_metadata = {
    "target_strategy": "calibrated_description_similarity_soft_targets",
    "concept_description_column": "description",
}

checkpoint_payload = {
    "model_state_dict": concept_qa_model.state_dict(),
    "embedding_dim": embedding_dim,
    "encoder_name": encoder_name,
    "concepts": concepts_df.to_dict(orient="records"),
    "concepts_path": str(concepts_path.relative_to(repo_root)),
    "concept_embeddings": concept_embeddings.detach().cpu(),
    "concept_dictionary": concept_dictionary.detach().cpu(),
    "calibration_medians": concept_medians.detach().cpu(),
    "calibration_stds": concept_stds.detach().cpu(),
    "calibration_temperature": calibration_temperature,
    "target_metadata": target_metadata,
    "test_metrics": concept_qa_test_metrics,
}

torch.save(checkpoint_payload, qa_checkpoint)

if save_split_cache:
    cache_payload = {
        "encoder_name": encoder_name,
        "embedding_dim": embedding_dim,
        "concepts": concepts_df.to_dict(orient="records"),
        "concept_embeddings": concept_embeddings.detach().cpu(),
        "concept_dictionary": concept_dictionary.detach().cpu(),
        "calibration_medians": concept_medians.detach().cpu(),
        "calibration_stds": concept_stds.detach().cpu(),
        "calibration_temperature": calibration_temperature,
        "target_metadata": target_metadata,
        "splits": {
            split_name: {
                "embeddings": split_data["embeddings"].detach().cpu(),
                "similarity_scores": split_data["similarity_scores"].detach().cpu(),
                "calibrated_soft_targets": split_data["calibrated_soft_targets"].detach().cpu(),
                "profession_ids": split_data["profession_ids"].cpu(),
                "gender_ids": split_data["gender_ids"].cpu(),
                "profession_labels": split_data["profession_labels"],
                "gender_labels": split_data["gender_labels"],
            }
            for split_name, split_data in encoded_splits.items()
        },
    }
    torch.save(cache_payload, qa_cache_path)

history_payload = {
    "history": concept_qa_history,
    "test_metrics": concept_qa_test_metrics,
    "target_metadata": target_metadata,
}
with open(qa_history_path, "w", encoding="utf-8") as handle:
    json.dump(history_payload, handle, indent=2)

{
    "checkpoint": str(qa_checkpoint.relative_to(repo_root)),
    "history": str(qa_history_path.relative_to(repo_root)),
    "cache": str(qa_cache_path.relative_to(repo_root)) if save_split_cache else None,
}